In [519]:
import numpy as np
import math
import pandas as pd
import cv2
import os
#import tqdm
#from scipy.io import loadmat

In [520]:
stock_data = []

stock_folder = '/Users/subrata/workstation/jupyterFiles/stock_market_analysis/equity_list_1' + '.csv'
stock_detail = pd.read_csv(stock_folder)
stock_data.append(stock_detail)
    
equity_data_1 = pd.concat(stock_data)

stock_data = []

stock_folder = '/Users/subrata/workstation/jupyterFiles/stock_market_analysis/equity_list_2' + '.xlsx'
stock_detail = pd.read_excel(stock_folder)
stock_data.append(stock_detail)
    
equity_data_2 = pd.concat(stock_data)

stock_data = []

stock_folder = '/Users/subrata/workstation/jupyterFiles/stock_market_analysis/equity_list_3' + '.xls'
stock_detail = pd.read_excel(stock_folder)
stock_data.append(stock_detail)
    
equity_data_3 = pd.concat(stock_data)

In [521]:
equity_data_1.head()

,SYMBOL,NAME OF COMPANY,SERIES,DATE OF LISTING,PAID UP VALUE,MARKET LOT,ISIN NUMBER,FACE VALUE
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,5.0,1,INE144J01027,5
1,21STCENMGM,21st Century Management Services Limited,BE,03-MAY-1995,10.0,1,INE253B01015,10
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1.0,1,INE466L01038,1
3,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,10.0,1,INE748C01038,10
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,10.0,1,INE470A01017,10


In [522]:
equity_data_2.head()

,Sr. No.,Symbol,Company Name,"Market capitalization as on March 28, 2024\n(In lakhs)"
0,1,RELIANCE,Reliance Industries Limited,201056022.448876
1,2,TCS,Tata Consultancy Services Limited,140247926.460234
2,3,HDFCBANK,HDFC Bank Limited,109991524.325625
3,4,ICICIBANK,ICICI Bank Limited,76765684.662095
4,5,BHARTIARTL,Bharti Airtel Limited,69478399.828022


In [523]:
equity_data_3.head()

,Company,Current price,52W high/low,Market cap,Industry
0,20 Microns Ltd. 20MICRONS,₹189.53 +4.58%,₹203.00/₹90.09,₹670.44 Crs,Mining/Minerals
1,360 One Wam Ltd. 360ONE,₹802.10 +2.54%,₹900.95/₹419.85,₹28972.93 Crs,Finance & Investments
2,3M India Ltd. 3MINDIA,₹36489.25 -1.10%,₹39876.09/₹26512.00,₹41105.11 Crs,Diversified
3,3P Land Holdings Ltd. 3PLAND,₹31.05 -1.11%,₹40.00/₹18.35,₹55.8 Crs,Finance & Investments
4,3i Infotech Ltd. 3IINFOLTD,₹36.71 -1.37%,₹63.89/₹30.25,₹626.28 Crs,IT Consulting & Software


In [524]:
equity_data_1.drop(' PAID UP VALUE', axis=1, inplace=True)
equity_data_1.rename(columns={'SYMBOL': 'symbol', 'NAME OF COMPANY': 'company_name_1', ' SERIES': 'series', ' DATE OF LISTING': 'date_of_listing', ' MARKET LOT': 'market_lot', ' ISIN NUMBER': 'isin_number', ' FACE VALUE': 'face_val' }, inplace=True)
equity_data_1['symbol'] = equity_data_1['symbol'].str.strip()

equity_data_1.head()

,symbol,company_name_1,series,date_of_listing,market_lot,isin_number,face_val
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,1,INE144J01027,5
1,21STCENMGM,21st Century Management Services Limited,BE,03-MAY-1995,1,INE253B01015,10
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1,INE466L01038,1
3,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,1,INE748C01038,10
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,1,INE470A01017,10


In [525]:


equity_data_2.drop('Sr. No.', axis=1, inplace=True)
equity_data_2.rename(columns={'Symbol': 'symbol', 'Company Name': 'company_name_2', 'Market capitalization as on March 28, 2024\n(In lakhs)':'market_cap_l'}, inplace=True)

# Take out the dataframe having no market cap values : basically this col. having non-numeric values
equity_data_2_r1 = equity_data_2[pd.to_numeric(equity_data_2['market_cap_l'], errors='coerce').isnull()]

# Creating dataframe having positive market cap i.e. subtract no market cap dataframe from main dataframe
merged = equity_data_2.merge(equity_data_2_r1, on=list(equity_data_2.columns), how='outer', indicator=True)
equity_data_2_final = merged[merged['_merge'] == 'left_only'].drop('_merge', axis=1)

equity_data_2_final['market_cap_cr_1'] = equity_data_2_final['market_cap_l'] * 0.01
equity_data_2_final.drop('market_cap_l', axis=1, inplace=True)

equity_data_2_final['symbol'] = equity_data_2_final['symbol'].str.strip()

equity_data_2_final.head()


,symbol,company_name_2,market_cap_cr_1
0,RELIANCE,Reliance Industries Limited,2010560.224489
1,TCS,Tata Consultancy Services Limited,1402479.264602
2,HDFCBANK,HDFC Bank Limited,1099915.243256
3,ICICIBANK,ICICI Bank Limited,767656.846621
4,BHARTIARTL,Bharti Airtel Limited,694783.99828


In [526]:
# Function to replace non-breaking space with a regular space

def replace_nbsp(s):
    return s.replace('\xa0', ' ')


In [527]:
# Function to break one column into two based on some criteria

import re

def extract_parts(s):
    # Regular expression to match the pattern
    pattern = r'(.+[. ]+)(\S+)$'
    match = re.search(pattern, s)
    if match:
        part1 = match.group(1)
        part2 = match.group(2)
        return part1, part2
    else:
        return s, None

In [528]:

equity_data_3['Company'] = equity_data_3['Company'].apply(replace_nbsp)

equity_data_3[['company_name_3', 'symbol']] = equity_data_3['Company'].apply(lambda s: pd.Series(extract_parts(s)))
equity_data_3.drop('Company', axis=1, inplace=True)

equity_data_3[['current_price', 'increase']] = equity_data_3['Current price'].str.split('[+|-]', 1, expand=True)
equity_data_3.drop('Current price', axis=1, inplace=True)
equity_data_3.drop('increase', axis=1, inplace=True)

equity_data_3[['52w_high', '52w_low']] = equity_data_3['52W high/low'].str.split('/', 1, expand=True)
equity_data_3.drop('52W high/low', axis=1, inplace=True)

# equity_data_3.head()

cols = ['symbol', 'company_name_3', 'Market cap', 'current_price', '52w_high', '52w_low', 'Industry']
equity_data_3 = equity_data_3[cols]

equity_data_3.rename(columns={'Market cap': 'market_cap_cr', 'Industry': 'industry'}, inplace=True)

equity_data_3['symbol'] = equity_data_3['symbol'].str.strip()

equity_data_3.head()

/var/folders/_y/wjshqn7x7lgf5rrlggyz8j3r0000gp/T/ipykernel_81276/933684969.py:6: FutureWarning: In a future version of pandas all arguments of StringMethods.split except for the argument 'pat' will be keyword-only.
  equity_data_3[['current_price', 'increase']] = equity_data_3['Current price'].str.split('[+|-]', 1, expand=True)
/var/folders/_y/wjshqn7x7lgf5rrlggyz8j3r0000gp/T/ipykernel_81276/933684969.py:10: FutureWarning: In a future version of pandas all arguments of StringMethods.split except for the argument 'pat' will be keyword-only.
  equity_data_3[['52w_high', '52w_low']] = equity_data_3['52W high/low'].str.split('/', 1, expand=True)


,symbol,company_name_3,market_cap_cr,current_price,52w_high,52w_low,industry
0,20MICRONS,20 Microns Ltd.,₹670.44 Crs,₹189.53,₹203.00,₹90.09,Mining/Minerals
1,360ONE,360 One Wam Ltd.,₹28972.93 Crs,₹802.10,₹900.95,₹419.85,Finance & Investments
2,3MINDIA,3M India Ltd.,₹41105.11 Crs,₹36489.25,₹39876.09,₹26512.00,Diversified
3,3PLAND,3P Land Holdings Ltd.,₹55.8 Crs,₹31.05,₹40.00,₹18.35,Finance & Investments
4,3IINFOLTD,3i Infotech Ltd.,₹626.28 Crs,₹36.71,₹63.89,₹30.25,IT Consulting & Software


In [529]:
# Remove ruppees sign and Crs from the columns

equity_data_3['market_cap_cr'] = equity_data_3['market_cap_cr'].str.replace(' Crs', '')

equity_data_3['market_cap_cr'] = equity_data_3['market_cap_cr'].str[1:]
equity_data_3['current_price'] = equity_data_3['current_price'].str[1:]
equity_data_3['52w_high'] = equity_data_3['52w_high'].str[1:]
equity_data_3['52w_low'] = equity_data_3['52w_low'].str[1:]

equity_data_3.head()

,symbol,company_name_3,market_cap_cr,current_price,52w_high,52w_low,industry
0,20MICRONS,20 Microns Ltd.,670.44,189.53,203.00,90.09,Mining/Minerals
1,360ONE,360 One Wam Ltd.,28972.93,802.10,900.95,419.85,Finance & Investments
2,3MINDIA,3M India Ltd.,41105.11,36489.25,39876.09,26512.00,Diversified
3,3PLAND,3P Land Holdings Ltd.,55.8,31.05,40.00,18.35,Finance & Investments
4,3IINFOLTD,3i Infotech Ltd.,626.28,36.71,63.89,30.25,IT Consulting & Software


In [530]:
# Merge the data frames on the common column with indicator=True
merged_df = equity_data_1.merge(equity_data_2_final, on='symbol', how='outer', indicator=True)

merged_df.head()

,symbol,company_name_1,series,date_of_listing,market_lot,isin_number,face_val,company_name_2,market_cap_cr_1,_merge
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,1.0,INE144J01027,5.0,20 Microns Limited,507.067034,both
1,21STCENMGM,21st Century Management Services Limited,BE,03-MAY-1995,1.0,INE253B01015,10.0,21st Century Management Services Limited,43.68,both
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1.0,INE466L01038,1.0,360 ONE WAM LIMITED,24236.81828,both
3,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,1.0,INE748C01038,10.0,3i Infotech Limited,667.07502,both
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,1.0,INE470A01017,10.0,3M India Limited,35139.527128,both


In [531]:
# Merge with the 3rd dataframe

merged_df.rename(columns={'_merge': '1st_merge'}, inplace=True)
all_merged_df = merged_df.merge(equity_data_3, on='symbol', how='outer', indicator=True)

all_merged_df.head()

,symbol,company_name_1,series,date_of_listing,market_lot,isin_number,face_val,company_name_2,market_cap_cr_1,1st_merge,company_name_3,market_cap_cr,current_price,52w_high,52w_low,industry,_merge
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,1.0,INE144J01027,5.0,20 Microns Limited,507.067034,both,20 Microns Ltd.,670.44,189.53,203.00,90.09,Mining/Minerals,both
1,21STCENMGM,21st Century Management Services Limited,BE,03-MAY-1995,1.0,INE253B01015,10.0,21st Century Management Services Limited,43.68,both,Twentyfirst Century Management Services Ltd.,58.8,55.75,55.75,17.69,Capital Markets Related Services,both
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1.0,INE466L01038,1.0,360 ONE WAM LIMITED,24236.81828,both,360 One Wam Ltd.,28972.93,802.10,900.95,419.85,Finance & Investments,both
3,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,1.0,INE748C01038,10.0,3i Infotech Limited,667.07502,both,3i Infotech Ltd.,626.28,36.71,63.89,30.25,IT Consulting & Software,both
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,1.0,INE470A01017,10.0,3M India Limited,35139.527128,both,3M India Ltd.,41105.11,36489.25,39876.09,26512.00,Diversified,both


In [532]:
cols_list = list(all_merged_df.columns)
print(cols_list)

['symbol', 'company_name_1', 'series', 'date_of_listing', 'market_lot', 'isin_number', 'face_val', 'company_name_2', 'market_cap_cr_1', '1st_merge', 'company_name_3', 'market_cap_cr', 'current_price', '52w_high', '52w_low', 'industry', '_merge']


In [533]:
#reorient columns

new_cols_list = ['symbol', 'company_name_1', 'company_name_2', 'company_name_3', 'series', 'date_of_listing', 'market_lot', 'isin_number', 'face_val', 'market_cap_cr_1', 'market_cap_cr', 'current_price', '52w_high', '52w_low', 'industry', '1st_merge', '_merge']
all_merged_df = all_merged_df[new_cols_list]
all_merged_df.head()

,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge
0,20MICRONS,20 Microns Limited,20 Microns Limited,20 Microns Ltd.,EQ,06-OCT-2008,1.0,INE144J01027,5.0,507.067034,670.44,189.53,203.00,90.09,Mining/Minerals,both,both
1,21STCENMGM,21st Century Management Services Limited,21st Century Management Services Limited,Twentyfirst Century Management Services Ltd.,BE,03-MAY-1995,1.0,INE253B01015,10.0,43.68,58.8,55.75,55.75,17.69,Capital Markets Related Services,both,both
2,360ONE,360 ONE WAM LIMITED,360 ONE WAM LIMITED,360 One Wam Ltd.,EQ,19-SEP-2019,1.0,INE466L01038,1.0,24236.81828,28972.93,802.10,900.95,419.85,Finance & Investments,both,both
3,3IINFOLTD,3i Infotech Limited,3i Infotech Limited,3i Infotech Ltd.,EQ,22-OCT-2021,1.0,INE748C01038,10.0,667.07502,626.28,36.71,63.89,30.25,IT Consulting & Software,both,both
4,3MINDIA,3M India Limited,3M India Limited,3M India Ltd.,EQ,13-AUG-2004,1.0,INE470A01017,10.0,35139.527128,41105.11,36489.25,39876.09,26512.00,Diversified,both,both


In [534]:
# final all equities dataframe

all_merged_equities = all_merged_df.copy().reset_index(drop=True)
print('Total no. of equities = ', len(all_merged_equities))
all_merged_equities.tail()

Total no. of equities =  2476


,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge
2471,VIKASWSP,NaN,NaN,Vikas WSP Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,20.44,1.44,3.39,0.94,Agricultural Products,NaN,right_only
2472,VISHAL,NaN,NaN,Vishal Fabrics Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,434.74,22.05,26.1,14.19,Textiles - Processing/Texturising,NaN,right_only
2473,WATERBASE,NaN,NaN,Waterbase Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,310.7,74.9,93.15,56.1,Aquaculture - Integrated,NaN,right_only
2474,WINPRO,NaN,NaN,WinPro Industries Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,29.99,2.6,6.4,2.14,Advertising & Media Agency,NaN,right_only
2475,LtdYAARI,NaN,NaN,Yaari Digital Integrated Services,NaN,NaN,NaN,NaN,NaN,NaN,110.49,10.75,15.65,6.7,IT Consulting & Software,NaN,right_only


In [535]:
# Define a function to check for the substrings
def contains_limited_or_ltd(symbol):
    return bool(re.search(r'^(Limited|Ltd|limited|ltd)', symbol, re.IGNORECASE))

# Apply the function to the 'symbol' column and create a new column 'contains_limited_or_ltd'
all_merged_equities['contains_limited_or_ltd'] = all_merged_equities['symbol'].apply(contains_limited_or_ltd)

# Filter rows that contain 'Limited', 'limited', 'Ltd', or 'ltd'
filtered_df = all_merged_equities[all_merged_equities['contains_limited_or_ltd']]

print(len(filtered_df))
filtered_df.tail()

#outcome result of this cell is only one row. Hence we are ignoring this.

1


,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge,contains_limited_or_ltd
2475,LtdYAARI,NaN,NaN,Yaari Digital Integrated Services,NaN,NaN,NaN,NaN,NaN,NaN,110.49,10.75,15.65,6.7,IT Consulting & Software,NaN,right_only,True


In [536]:
# Check whether or how many Nil values in company_name_1

all_merged_equities[all_merged_equities['company_name_1'].isnull()]

,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge,contains_limited_or_ltd
2134,MCDOWELL-N,NaN,United Spirits Limited,NaN,NaN,NaN,NaN,NaN,NaN,82499.770502,NaN,NaN,NaN,NaN,NaN,right_only,left_only,False
2135,L&TFH,NaN,L&T Finance Holdings Limited,NaN,NaN,NaN,NaN,NaN,NaN,39387.480406,NaN,NaN,NaN,NaN,NaN,right_only,left_only,False
2136,AEGISCHEM,NaN,Aegis Logistics Limited,NaN,NaN,NaN,NaN,NaN,NaN,15677.415,NaN,NaN,NaN,NaN,NaN,right_only,left_only,False
2137,UJJIVAN,NaN,Ujjivan Financial Services Limited,NaN,NaN,NaN,NaN,NaN,NaN,5801.821107,NaN,NaN,NaN,NaN,NaN,right_only,left_only,False
2138,BCG,NaN,Brightcom Group Limited,Brightcom Group Ltd.,NaN,NaN,NaN,NaN,NaN,2784.732185,1816.67,9.38,14.55,8.65,IT Consulting & Software,right_only,both,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2471,VIKASWSP,NaN,NaN,Vikas WSP Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,20.44,1.44,3.39,0.94,Agricultural Products,NaN,right_only,False
2472,VISHAL,NaN,NaN,Vishal Fabrics Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,434.74,22.05,26.1,14.19,Textiles - Processing/Texturising,NaN,right_only,False
2473,WATERBASE,NaN,NaN,Waterbase Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,310.7,74.9,93.15,56.1,Aquaculture - Integrated,NaN,right_only,False
2474,WINPRO,NaN,NaN,WinPro Industries Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,29.99,2.6,6.4,2.14,Advertising & Media Agency,NaN,right_only,False


In [537]:
# Replace NaN values in 'company_name_1' with corresponding 'company_name_2' values, then with 'company_name_3' values
all_merged_equities['company_name_1'] = all_merged_equities['company_name_1'].fillna(all_merged_equities['company_name_2']).fillna(all_merged_equities['company_name_3'])

# Check whether above operation has been done successfully 
all_merged_equities[all_merged_equities['company_name_1'].isnull()]


,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge,contains_limited_or_ltd


In [539]:
# Check whether any 'symbol' is Nil

all_merged_equities[all_merged_df['symbol'].isnull()]

,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge,contains_limited_or_ltd


In [547]:
# Replace nil values in 'market_cap_cr' with corresponding values of 'market_cap_cr_1

all_merged_equities['market_cap_cr'] = all_merged_equities['market_cap_cr'].fillna(all_merged_equities['market_cap_cr_1'])

# Check whether above operation has been done successfully 
all_merged_equities[all_merged_equities['market_cap_cr'].isnull()]


,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge,contains_limited_or_ltd
280,BOROSCI,Borosil Scientific Limited,NaN,NaN,BE,07-JUN-2024,1.0,INE02L001032,1.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
515,ESSEN-RE2,Integra Essen Ltd-RE,NaN,NaN,BE,11-JUN-2024,1.0,INE418N20035,1.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
537,FELDVR,Future Enterprises Limited,NaN,NaN,BE,13-FEB-2009,1.0,IN9623B01058,2.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
576,GATECHDVR,GACM Technologies Limited,NaN,NaN,BE,10-OCT-2017,1.0,INE224E01036,1.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
967,JISLDVREQS,Jain Irrigation Systems Limited,NaN,NaN,EQ,30-NOV-2011,1.0,IN9175A01010,2.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
1132,KRONOX,Kronox Lab Sciences Limited,NaN,NaN,BE,10-JUN-2024,1.0,INE0ATZ01017,10.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
1181,LTF,L&T Finance Limited,NaN,NaN,EQ,12-AUG-2011,1.0,INE498L01015,10.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
1365,NGIL-RE2,Nakoda Group of Industries Limited-RE,NaN,NaN,BE,13-JUN-2024,1.0,INE236Y20038,10.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False
1900,TATAMTRDVR,Tata Motors Limited,NaN,NaN,EQ,05-NOV-2008,1.0,IN9155A01020,2.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only,left_only,False


In [564]:
all_merged_equities.loc[280, 'market_cap_cr'] = 1521.08
all_merged_equities.loc[280, 'current_price'] = 171.3
all_merged_equities.loc[280, '52w_high'] = 173.15
all_merged_equities.loc[280, '52w_low'] = 138.00

all_merged_equities.loc[515, 'market_cap_cr'] = 28.73
all_merged_equities.loc[515, 'current_price'] = 1.87
all_merged_equities.loc[515, '52w_high'] = 1.87
all_merged_equities.loc[515, '52w_low'] = 0.35

all_merged_equities.loc[537, 'market_cap_cr'] = 24.02
all_merged_equities.loc[537, 'current_price'] = 6.10
all_merged_equities.loc[537, '52w_high'] = 6.85
all_merged_equities.loc[537, '52w_low'] = 4.10

all_merged_equities.loc[576, 'market_cap_cr'] = 18.44
all_merged_equities.loc[576, 'current_price'] = 3.12
all_merged_equities.loc[576, '52w_high'] = 12.40
all_merged_equities.loc[576, '52w_low'] = 2.87

all_merged_equities.loc[967, 'market_cap_cr'] = 83.47
all_merged_equities.loc[967, 'current_price'] = 43.26
all_merged_equities.loc[967, '52w_high'] = 45.95
all_merged_equities.loc[967, '52w_low'] = 19.70

all_merged_equities.loc[1132, 'market_cap_cr'] = 560.27
all_merged_equities.loc[1132, 'current_price'] = 151.00
all_merged_equities.loc[1132, '52w_high'] = 165.6
all_merged_equities.loc[1132, '52w_low'] = 143.78

all_merged_equities.loc[1181, 'market_cap_cr'] = 45461.98
all_merged_equities.loc[1181, 'current_price'] = 182.59
all_merged_equities.loc[1181, '52w_high'] = 186.70
all_merged_equities.loc[1181, '52w_low'] = 114.95

all_merged_equities.loc[1365, 'market_cap_cr'] = 7.28
all_merged_equities.loc[1365, 'current_price'] = 14.30
all_merged_equities.loc[1365, '52w_high'] = 19.00
all_merged_equities.loc[1365, '52w_low'] = 8.50

all_merged_equities.loc[1900, 'market_cap_cr'] = 33029.81
all_merged_equities.loc[1900, 'current_price'] = 649.55
all_merged_equities.loc[1900, '52w_high'] = 712.60
all_merged_equities.loc[1900, '52w_low'] = 294.40


In [565]:
# Check whether above operation has been done successfully 
all_merged_equities[all_merged_equities['market_cap_cr'].isnull()]

,symbol,company_name_1,company_name_2,company_name_3,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr_1,market_cap_cr,current_price,52w_high,52w_low,industry,1st_merge,_merge,contains_limited_or_ltd


In [566]:

col_list = list(all_merged_equities.columns)
print(col_list)

['symbol', 'company_name_1', 'company_name_2', 'company_name_3', 'series', 'date_of_listing', 'market_lot', 'isin_number', 'face_val', 'market_cap_cr_1', 'market_cap_cr', 'current_price', '52w_high', '52w_low', 'industry', '1st_merge', '_merge', 'contains_limited_or_ltd']


In [567]:
equity_master = all_merged_equities[['symbol', 'company_name_1', 'series', 'date_of_listing', 'market_lot', 'isin_number', 'face_val', 'market_cap_cr', 'current_price', '52w_high', '52w_low', 'industry']]
equity_master.head()

,symbol,company_name_1,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr,current_price,52w_high,52w_low,industry
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,1.0,INE144J01027,5.0,670.44,189.53,203.00,90.09,Mining/Minerals
1,21STCENMGM,21st Century Management Services Limited,BE,03-MAY-1995,1.0,INE253B01015,10.0,58.8,55.75,55.75,17.69,Capital Markets Related Services
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1.0,INE466L01038,1.0,28972.93,802.10,900.95,419.85,Finance & Investments
3,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,1.0,INE748C01038,10.0,626.28,36.71,63.89,30.25,IT Consulting & Software
4,3MINDIA,3M India Limited,EQ,13-AUG-2004,1.0,INE470A01017,10.0,41105.11,36489.25,39876.09,26512.00,Diversified


In [568]:
equity_master.tail()

,symbol,company_name_1,series,date_of_listing,market_lot,isin_number,face_val,market_cap_cr,current_price,52w_high,52w_low,industry
2471,VIKASWSP,Vikas WSP Ltd.,NaN,NaN,NaN,NaN,NaN,20.44,1.44,3.39,0.94,Agricultural Products
2472,VISHAL,Vishal Fabrics Ltd.,NaN,NaN,NaN,NaN,NaN,434.74,22.05,26.1,14.19,Textiles - Processing/Texturising
2473,WATERBASE,Waterbase Ltd.,NaN,NaN,NaN,NaN,NaN,310.7,74.9,93.15,56.1,Aquaculture - Integrated
2474,WINPRO,WinPro Industries Ltd.,NaN,NaN,NaN,NaN,NaN,29.99,2.6,6.4,2.14,Advertising & Media Agency
2475,LtdYAARI,Yaari Digital Integrated Services,NaN,NaN,NaN,NaN,NaN,110.49,10.75,15.65,6.7,IT Consulting & Software


In [571]:
stock_data = []

stock_folder = '/Users/subrata/workstation/jupyterFiles/stock_market_analysis/equity_list_4' + '.xls'
stock_detail = pd.read_excel(stock_folder)
stock_data.append(stock_detail)
    
equity_data_4 = pd.concat(stock_data)

In [572]:
equity_data_4.head()

,Company Name,Promoter Holding %,FII Holding %,DII Holding %,Promoter Pledge %,Prom Hold YoY Chg %,Prom Pledge YoY Chg,Market Cap (Rs Cr),PE TTM
0,BLS Infotech,0.591,-,-,-,-,-,123.0,32097.62
1,Tinna Trade,0.738,-,-,-,-,-,1313.0,11192.02
2,Thirdwave Financial,0.669,-,0.0617,-,0.212,-,395.0,9988.43
3,Milgrey Finance,0.241,-,-,0.111,-0.2876,-0.4363,136.0,6506.42
4,Justride Enterprises,0.44,-,-,-,-0.2908,-,1112.0,3244.50


In [574]:
print(equity_data_4.columns)

Index(['Company Name', 'Promoter Holding %', 'FII Holding %', 'DII Holding %',
       'Promoter Pledge %', 'Prom Hold YoY Chg %', 'Prom Pledge YoY Chg',
       'Market Cap (Rs Cr)', 'PE TTM'],
      dtype='object')


In [576]:
equity_pe_master = equity_data_4.copy()

equity_pe_master.drop(['Promoter Pledge %', 'Prom Hold YoY Chg %', 'Prom Pledge YoY Chg'], axis=1, inplace=True)
equity_pe_master.rename(columns={'Company Name': 'company_name', 'Promoter Holding %': 'promoter_holding', 'FII Holding %': 'fii_holding', 'DII Holding %': 'dii_holding', 'Market Cap (Rs Cr)': 'market_cap', 'PE TTM': 'ttm_pe'}, inplace=True)

equity_pe_master.head()


,company_name,promoter_holding,fii_holding,dii_holding,market_cap,ttm_pe
0,BLS Infotech,0.591,-,-,123.0,32097.62
1,Tinna Trade,0.738,-,-,1313.0,11192.02
2,Thirdwave Financial,0.669,-,0.0617,395.0,9988.43
3,Milgrey Finance,0.241,-,-,136.0,6506.42
4,Justride Enterprises,0.44,-,-,1112.0,3244.50


In [ ]:
import yfinance as yf
import pandas as pd

# List of some popular Indian stocks (you can expand this list)
symbols = [
    'TCS.NS', 'INFY.NS', 'RELIANCE.NS', 'HDFCBANK.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'SBIN.NS', 'AXISBANK.NS', 'KOTAKBANK.NS', 'BAJFINANCE.NS',
    'HCLTECH.NS', 'ASIANPAINT.NS', 'ITC.NS', 'LT.NS', 'MARUTI.NS',
    'SUNPHARMA.NS', 'WIPRO.NS', 'NTPC.NS', 'POWERGRID.NS', 'ONGC.NS'
]

# Fetch data
data = {}
for symbol in symbols:
    stock = yf.Ticker(symbol)
    info = stock.info
    data[symbol] = {
        'name': info['longName'],
        'symbol': symbol,
        'peRatio': info.get('trailingPE', None)
    }

# Convert to DataFrame
df = pd.DataFrame.from_dict(data, orient='index')

# Filter companies with P/E ratio greater than 50
high_pe_companies = df[df['peRatio'] > 50]

# Print the result
print(high_pe_companies)
